In [13]:
from openai import OpenAI, OpenAIError
api_key = "sk-j7gjDwdfs0xu21OYA9139e05B52d4aF6B6A2E5Bd0cA9C7Cb"
client = OpenAI(
    api_key=api_key,
    base_url="https://fast.xeduapi.com/v1"
)

generate_func_docstring = """你是一个代码解释器，你需要按照下面给出的模板解读一个函数的功能和作用，以及输入参数和返回值解读。

## 解读模板
函数作用
args：
    para1(参数类型)：参数含义
    para2(参数类型)：参数含义
return:
    返回值解读

{code}
"""

generate_cell_md = """你是一个代码解释器，你需要按照下面给出的模板解读代码的功能，作用和难点分析。要求以markdown格式返回。

# 要求
1. 根据这段代码的核心功能生成一级标题
2. 根据这段代码的不同作用生成二级标题，正文则详细介绍代码作用
3. 对于代码中调用的重要函数，使用docstring的文档注释规范进行解读。
4. 代码难点作为单独的二级标题，将每个难点作为三级标题给出，并详细介绍难点内容。
5. 使用中文进行解读。

需要解读的代码为：{code}

解读："""

generate_code_line = """你是一个代码解释器，你需要逐行解读下面给出的代码片段，解读每一行代码的作用和含义，要求在每一行代码的前面新增一行，用#号注释解读内容。

{code}"""
def run_gpt(cell_content, prompt=generate_cell_md):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt.format(code=cell_content),
            }
        ]
    )
    return response.choices[0].message.content

# print(run_gpt("""response = client.chat.completions.create(
#                 model=model_name,
#                 messages=[
#                     {
#                         "role": "user",
#                         "content": prompts,
#                     }
#                 ],
#                 max_tokens=target_length,
#                 temperature=temperature,
#                 top_p=top_p,
#                 frequency_penalty=frequency_penalty,
#                 presence_penalty=presence_penalty,
#                 stop=stop_sequences,
#                 logprobs=logprobs,
#                 n=n
#             )"""))

In [14]:
import secrets
def generate_cell_id(notebook_ids):
    nid = secrets.token_hex(4)
    while nid in notebook_ids:
        nid = secrets.token_hex(4)
    return nid

print(generate_cell_id([]))

c63001f6


In [15]:
def parse_notebook_content(notebook_cells):
    res_notebook_cells = []
    notebook_ids = [c["id"] for c in notebook_cells]
    for cid, cell in enumerate(notebook_cells):
        if cell["cell_type"]=="code":
            cell_code = "".join(cell["source"])
            interprete = run_gpt(cell_content=cell_code, prompt=generate_cell_md)
            if interprete.startswith("```markdown"): interprete = interprete[len("```markdown"):]
            if interprete.endswith("```"): interprete = interprete[:-3]
            nid = generate_cell_id(notebook_ids)
            notebook_ids.append(nid)
            res_notebook_cells.append({
                "cell_type": "markdown",
                "id": nid,
                "metadata": {},
                "source": [interprete]
            })
            res_notebook_cells.append(cell)
        else:
            res_notebook_cells.append(cell)

    return res_notebook_cells

In [16]:
import os
print(os.path.exists("/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/DPO/BT_model.ipynb"))

True


In [17]:
import json
from pathlib import Path

# notebook_files = ["/root/LLM-tutorial-AllinOne/repo/7_手撕LLM_手撕ALpaca-Chinese-LoRA_1116/Chinese-LLaMA-Alpaca/scripts/training/Chinese_LLaMA_Alpaca.ipynb"]
notebook_files = []
for p in Path("/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/").iterdir():
    for s in p.rglob('*.ipynb'):  
         # yield s
        s = str(s)
        if s.endswith("_inter.ipynb"): continue
        notebook_files.append(s)

for nf in notebook_files:
    print(f"Start: {nf} ")
    if os.path.exists(nf.replace(".ipynb", "_inter.ipynb")): continue
    with open(nf, 'r', encoding='utf-8') as f:
        notebook_data = json.load(f)
    inter_notebook_data = parse_notebook_content(notebook_data["cells"])
    notebook_data["cells"] = inter_notebook_data   
    with open(nf.replace(".ipynb", "_inter.ipynb"), 'w', encoding='utf-8') as f:
        json.dump(notebook_data, f, ensure_ascii=False) 
    print(f"Finish:{nf}")

        

Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/DPO/BT_model.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/DPO/DPO.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/KTO/KTO.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/AdamW-pytorch.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/BPE-Tokenizer.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/Cross_Entropy.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/Cross_Entropy_gradient.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/Perplexity-Pytorch.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/Softmax-Implemention.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/Tensor_Parallelism.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/attention_score_why_scaled_with_sqrt_d.ipynb 
Start: /root/LLM-tutorial-AllinOne/repo/MA-RLHF/

In [18]:
import os
import glob

# 获取当前目录
directory = "/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook"

# 获取所有文件
files = glob.glob(directory + "/*")

# 输出所有文件名
for file in files:
    print(file)


/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/DPO
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/Easy_generation_with_kvcache.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/GPT-loss.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/KTO
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/LLM_Pipeline_Fintune_LLaMA2_QLoRA_RLHF_20240318.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/LayerNorm.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/LayerNorm_and_RMSNorm_analysis.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/Llama3-GQA.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/README.md
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/RLHF_PPO_Pytorch.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/beam_search.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/flashattention
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/format_datasets.ipynb
/root/LLM-tutorial-Alli

In [19]:
from pathlib import Path

for p in Path("/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/").iterdir():
    for s in p.rglob('*.ipynb'):  
         # yield s
          print(s)

# 这样就可以获取到所有嵌套文件的路径

/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/DPO/BT_model.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/DPO/BT_model_inter.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/DPO/DPO.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/DPO/DPO_inter.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/KTO/KTO.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/KTO/KTO_inter.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/AdamW-pytorch.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/AdamW-pytorch_inter.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/BPE-Tokenizer.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/BPE-Tokenizer_inter.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/Cross_Entropy.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/Cross_Entropy_gradient.ipynb
/root/LLM-tutorial-AllinOne/repo/MA-RLHF/notebook/common/Cross_Entropy_gradient_inter.ipynb
/root/LLM-tut